## 셀 1. 패키지 설치 (최초 1회)

In [1]:
# 최초 1회만 실행.
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers accelerate qwen-vl-utils
!pip install -q flash-attn --no-build-isolation
!pip install -q Pillow opencv-python-headless

print("설치 완료!")

설치 완료!


## 셀 2. 모델 로딩 (최초 1회)

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import time, re, json, os, glob
import os
os.environ["HF_TOKEN"] = ""

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"모델 로딩 중: {MODEL_NAME} ...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

print(f"모델 로딩 완료! (VRAM 사용: {torch.cuda.memory_allocated()/1e9:.1f} GB)")

GPU: NVIDIA GeForce RTX 5090
VRAM: 33.7 GB
모델 로딩 중: Qwen/Qwen2.5-VL-7B-Instruct ...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

모델 로딩 완료! (VRAM 사용: 16.6 GB)


## 셀 3. 전성분 추출 (반복 사용)

`IMAGE_PATHS`에 이미지 경로를 넣고 실행

In [3]:
IMAGE_PATHS = [
    "/workspace/ocr 테스트.jpg",
    "/workspace/ocr_second_test.jpg",
    "/workspace/ocr_test_third.jpg",
]

PROMPT = """이 화장품 이미지에서 전성분(ingredients) 목록만 추출해주세요.

규칙:
1. "전성분" 또는 "[전성분]" 라벨 뒤에 나오는 성분들만 추출하세요.
2. 각 성분을 쉼표(,)로 구분하여 한 줄로 나열하세요.
3. 마케팅 문구, 사용방법, 주의사항, 제품명, 브랜드명은 절대 포함하지 마세요.
4. 성분명은 이미지에 적힌 한국어 표기 그대로 적어주세요.
5. 전성분이 보이지 않으면 "NOT_FOUND"라고만 답하세요.
6. 다른 설명 없이 성분 목록만 출력하세요.

출력 형식 예시:
정제수, 글리세린, 나이아신아마이드, 부틸렌글라이콜, 판테놀"""

NON_INGREDIENT = [
    r'피부를?\s*(탱탱|촉촉|건강|밝)', r'효과', r'사용\s*방법', r'HOW\s*TO',
    r'품번', r'품명', r'제조', r'주의\s*사항', r'눈으로\s*보이는',
    r'녹아들', r'밀착하여', r'간편하게', r'\d+min', r'마스크가',
]

def extract_ingredients(image_path):
    """이미지 한 장에서 전성분 추출"""
    t0 = time.time()
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text", "text": PROMPT},
        ],
    }]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=1024, temperature=0.1, do_sample=False
        )
    
    generated = output_ids[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(generated, skip_special_tokens=True)[0]
    elapsed = time.time() - t0
    
    # 파싱
    if "NOT_FOUND" in response:
        return {"file": os.path.basename(image_path), "status": "not_found",
                "ingredients": [], "count": 0, "time": round(elapsed, 2), "raw": response}
    
    parts = re.split(r'[,\n]', response)
    ingredients = []
    seen = set()
    for p in parts:
        c = p.strip().strip('- ·•')
        c = re.sub(r'^\d+[\.)\s]+', '', c).strip()
        if len(c) < 2:
            continue
        if any(re.search(pat, c) for pat in NON_INGREDIENT):
            continue
        if re.match(r'^[\d\s]+$', c):
            continue
        key = c.replace(' ', '').lower()
        if key in seen:
            continue
        seen.add(key)
        ingredients.append(c)
    
    return {
        "file": os.path.basename(image_path),
        "status": "success",
        "ingredients": ingredients,
        "count": len(ingredients),
        "time": round(elapsed, 2),
        "raw": response,
    }

if not IMAGE_PATHS:
    print("  IMAGE_PATHS 리스트에 이미지 경로를 넣어주세요!")
    print("    예: IMAGE_PATHS = ['/workspace/라로슈포제.jpg']")
else:
    results = []
    for path in IMAGE_PATHS:
        if not os.path.exists(path):
            print(f"파일 없음: {path}")
            continue
        print(f"{os.path.basename(path)}")
        
        r = extract_ingredients(path)
        results.append(r)
        
        print(f"  {r['time']}초 | 성분 {r['count']}개")
        for i, ing in enumerate(r['ingredients'], 1):
            print(f"  {i:3d}. {ing}")
    
    print(f"\n{'='*50}")
    print("요약")
    print(f"{'='*50}")
    for r in results:
        print(f"  {r['file']:25s} → {r['count']:3d}개 ({r['time']}초)")

ocr 테스트.jpg


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  6.77초 | 성분 19개
    1. 정제수
    2. 글리세린
    3. 폴리실리케이트-20
    4. 다이소듐이디티에이
    5. 세테리모늄브로마이드
    6. 나이아신아마이드
    7. 자일릴로
    8. 알란토인
    9. 프로폴리올리고
   10. 시카리아이드
   11. 만니톨
   12. 헥실데칸올
   13. 소듐에이드로사이드
   14. 시트리에씨드
   15. 람노스
   16. 사고씨주유물
   17. 유제시테
   18. 토코페롤
   19. 향료
ocr_second_test.jpg
  12.87초 | 성분 35개
    1. 정제수
    2. 글리세린
    3. 세틸에틸헥사노에이트
    4. 시어버터
    5. 아크릴레이트코폴리머
    6. 아이리쉬모스추출물
    7. 캐롭검
    8. 다이프로필렌글라이콜
    9. 베타인
   10. 포타슘클로라이드
   11. 하이드록시아세토페논
   12. 수크로오스
   13. 세틸하이드록시에틸셀룰로오스
   14. 카프릴릴글라이콜
   15. 알란토인
   16. 에틸헥실글리세린
   17. 폴리글리세릴-6카프릴레이트
   18. 폴리글리세릴-4카프레이트
   19. 아데노신
   20. 하이드롤라이즈드식물성단백질(100ppb)
   21. 2-헥산다이올
   22. 다이소듐이디티에이
   23. 향료
   24. 울트라마린 (CI 77007)
   25. 하이드레이티드실리카
   26. 다이소듐포스페이트
   27. 실리카다이메틸실릴레이트
   28. 다이포타슘글리시리제이트
   29. 부틸렌글라이콜
   30. 소듐포스페이트
   31. 녹두싹추출물
   32. 브로콜리싹추출물
   33. 유채싹추출물
   34. 적색227호 (CI 17200)
   35. 토코페롤
ocr_test_third.jpg
  14.48초 | 성분 38개
    1. 정제수
    2. 다이카프릴릴에터
    3. 판테놀
    4. 글리세린
    5. 펜틸렌글라이콜
    6.

## 셀 4. 결과 저장 (선택)

In [4]:
# JSON으로 저장
OUTPUT_PATH = "/workspace/extraction_results.json"

if 'results' in dir() and results:
    save_data = []
    for r in results:
        save_data.append({
            "file": r["file"],
            "status": r["status"],
            "ingredients": r["ingredients"],
            "ingredient_count": r["count"],
            "processing_time": r["time"],
        })
    
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    
    print(f"저장 완료: {OUTPUT_PATH}")
    print(f"총 {len(save_data)}개 제품, {sum(r['count'] for r in results)}개 성분")
else:
    print("먼저 셀 3을 실행하세요!")

저장 완료: /workspace/extraction_results.json
총 3개 제품, 92개 성분


---
## 🔧 빠른 테스트 (이미지 1장)
경로만 바꿔서 바로 테스트할 수 있습니다.

In [5]:
# 이미지 1장 빠른 테스트
TEST_IMAGE = "/workspace/ocr 테스트.jpg"  # ← 경로 수정

if os.path.exists(TEST_IMAGE):
    r = extract_ingredients(TEST_IMAGE)
    print(f"{r['time']}초 | 성분 {r['count']}개\n")
    for i, ing in enumerate(r['ingredients'], 1):
        print(f"  {i}. {ing}")
    print(f"\n--- VLM 원본 응답 ---\n{r.get('raw', '')}")
else:
    print(f"파일 없음: {TEST_IMAGE}")

4.24초 | 성분 19개

  1. 정제수
  2. 글리세린
  3. 폴리실리케이트-20
  4. 다이소듐이디티에이
  5. 세테리모늄브로마이드
  6. 나이아신아마이드
  7. 자일릴로
  8. 알란토인
  9. 프로폴리올리고
  10. 시카리아이드
  11. 만니톨
  12. 헥실데칸올
  13. 소듐에이드로사이드
  14. 시트리에씨드
  15. 람노스
  16. 사고씨주유물
  17. 유제시테
  18. 토코페롤
  19. 향료

--- VLM 원본 응답 ---
정제수, 글리세린, 폴리실리케이트-20, 다이소듐이디티에이, 세테리모늄브로마이드, 나이아신아마이드, 자일릴로, 알란토인, 프로폴리올리고, 시카리아이드, 만니톨, 헥실데칸올, 소듐에이드로사이드, 시트리에씨드, 람노스, 사고씨주유물, 유제시테, 토코페롤, 향료
